In [ ]:
# ==========================================
# ENVIRONMENT SETUP & INSTALLATIONS
# ==========================================
# Install and import libraries
%pip install pyarrow fastparquet geopandas shapely -q

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import os
import gc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 42.1 MB/s eta 0:00:00


In [ ]:
# ==========================================
# MOUNT GOOGLE DRIVE
# ==========================================
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL)
# PART 1 (RESIDENTIAL) - PHASE 1A PRE-CHECK: TABULAR DATA CHECK
# ==========================================
import pandas as pd

# Define Path
residential_parquet_path = '/content/drive/MyDrive/MSc_Dissertation_Project/02_processed_data/01_cleaned/residential_cleaned.parquet'

# Load the dataset
df_check = pd.read_parquet(residential_parquet_path)

# Verify Dimensions
print(f"Total Dimensions: {df_check.shape[0]:,} rows and {df_check.shape[1]} columns.")

# Audit Address Fields (Crucial for EPC String Matching)
print("\nAddress Field Missing Value Audit:")
print(df_check[['PAON', 'SAON', 'Street', 'Postcode']].isna().sum())

# Display the first 5 rows to verify string structure
print("\nFirst 5 rows of the dataset:")
print(df_check[['Transaction_ID', 'PAON', 'SAON', 'Street', 'Postcode']].head())

# Memory footprint check
memory_usage = df_check.memory_usage(deep=True).sum() / (1024**2)
print(f"\nMemory Usage: {memory_usage:.2f} MB")
print("Tabular data successfully verified. Ready to initiate the Spatial Bridge.")

Total Dimensions: 165,614 rows and 11 columns.

Address Field Missing Value Audit:
PAON             0
SAON        144989
Street       10000
Postcode        36
dtype: int64

First 5 rows of the dataset:
                           Transaction_ID                          PAON  SAON  \
0  {4C1B673F-0132-D8EA-E063-4704A8C0BF7D}                SCOTTS COTTAGE     2   
1  {4C1B673F-025A-D8EA-E063-4704A8C0BF7D}  LOWER GROUND FLOOR FLAT, 121  None   
2  {4C1B673F-025D-D8EA-E063-4704A8C0BF7D}                           19A  None   
3  {4C1B673F-025E-D8EA-E063-4704A8C0BF7D}                            25  None   
4  {4C1B673F-0260-D8EA-E063-4704A8C0BF7D}                           114  None   

            Street  Postcode  
0    MILLWAY PLACE   PL9 7RQ  
1  DURNFORD STREET   PL1 3QP  
2     TYRRELL MEAD  EX10 9TP  
3  WARDLOW GARDENS   PL6 5PU  
4    PLYMOUTH ROAD  TQ11 0DB  

Memory Usage: 59.12 MB
Tabular data successfully verified. Ready to initiate the Spatial Bridge.


In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1A PRE-CHECK: EPC DATASET PREVIEW
# ==========================================
import pandas as pd

print("Executing Memory-Safe EPC Database Preview...\n")

# Define Path
epc_csv_path = '/content/drive/MyDrive/MSc_Dissertation_Project/01_raw_data/property/EPC/EPC_Devon_County.csv'

# Read ONLY the first 5 rows to inspect headers and data types without crashing RAM
try:
    df_epc_preview = pd.read_csv(epc_csv_path, nrows=5)

    print(f"Total Columns Detected: {len(df_epc_preview.columns)}")
    print("\nExact Column Headers (First 20):")
    print(df_epc_preview.columns.tolist()[:20])

    print("\nFirst 5 rows of data (Address & Key Variables):")
    # Using a list comprehension to safely grab our target columns regardless of exact case
    target_cols = [col for col in df_epc_preview.columns if col.upper() in ['ADDRESS1', 'POSTCODE', 'UPRN', 'TOTAL_FLOOR_AREA']]
    print(df_epc_preview[target_cols].head())

except FileNotFoundError:
    print(f"ERROR: File not found at {epc_csv_path}. Please check your path.")

Executing Memory-Safe EPC Database Preview...

Total Columns Detected: 93

Exact Column Headers (First 20):
['certificate_number', 'address1', 'address2', 'address3', 'address', 'postcode', 'inspection_date', 'uprn', 'environment_impact_potential', 'energy_consumption_current', 'energy_consumption_potential', 'environment_impact_current', 'co2_emissions_current', 'co2_emiss_curr_per_floor_area', 'co2_emissions_potential', 'total_floor_area', 'lodgement_date', 'report_type', 'posttown', 'lodgement_datetime']

First 5 rows of data (Address & Key Variables):
            address1 postcode          uprn  total_floor_area
0  97 Alexandra Road  PL2 3BU  100040407163                56
1    20 Tamar Street  PL1 4FN   10070767977               108
2    18 Orchid Place  EX2 4FA   10096989370                71
3     9 Orchid Place  EX2 4FA   10096989361                51
4    29 Orchid Place  EX2 4FA   10096989381                51


In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1A: EPC UPRN & FLOOR AREA EXTRACTION
# ==========================================
import pandas as pd
import numpy as np
import gc
import os
import json

# Define Paths
residential_parquet_path = '/content/drive/MyDrive/MSc_Dissertation_Project/02_processed_data/01_cleaned/residential_cleaned.parquet'
epc_csv_path = '/content/drive/MyDrive/MSc_Dissertation_Project/01_raw_data/property/EPC/EPC_Devon_County.csv'
output_residential_uprn = '/content/drive/MyDrive/MSc_Dissertation_Project/02_processed_data/02_spatial/residential_with_uprn.parquet'
log_path = '/content/drive/MyDrive/MSc_Dissertation_Project/05_logs/script_logs/attrition_log.json'

# Force Colab to sync with the Google Drive folder to prevent OSError
os.makedirs(os.path.dirname(output_residential_uprn), exist_ok=True)

# Load the Cleaned Residential Tabular Data
df_res = pd.read_parquet(residential_parquet_path)

# Handle NaN values in addresses to prevent string corruption
df_res['SAON'] = df_res['SAON'].fillna('')
df_res['PAON'] = df_res['PAON'].fillna('')
df_res['Street'] = df_res['Street'].fillna('')

# onstruct Match Strings (Aggressively Normalized)
# Pass 1 string: Standard HMLR format (SAON + PAON + Street)
df_res['Match_String_1'] = (df_res['SAON'] + ' ' + df_res['PAON'] + ' ' + df_res['Street'])
df_res['Match_String_1'] = df_res['Match_String_1'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' +', ' ', regex=True).str.strip().str.upper()

# Pass 2 string (For Flats): Swapped format (PAON + SAON + Street)
df_res['Match_String_Swapped'] = (df_res['PAON'] + ' ' + df_res['SAON'] + ' ' + df_res['Street'])
df_res['Match_String_Swapped'] = df_res['Match_String_Swapped'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' +', ' ', regex=True).str.strip().str.upper()

# Create a highly optimized Set of our active postcodes to filter the EPC data during ingestion
devon_postcodes = set(df_res['Postcode'].unique())
print(f"Targeting {len(devon_postcodes):,} unique Devon postcodes for EPC matching.")

# Memory-Efficient EPC Ingestion
# Detect the lodgement-date column (header casing varies across EPC exports) so we can
# retain the MOST RECENT certificate per address rather than an arbitrary file-order row.
epc_header = pd.read_csv(epc_csv_path, nrows=0).columns
lodge_col = None
for _pref in ['lodgement_date', 'lodgement_datetime', 'inspection_date']:
    lodge_col = next((c for c in epc_header if c.strip().lower() == _pref), None)
    if lodge_col:
        break
if not lodge_col:
    print("WARNING: no lodgement/inspection date column found; falling back to file-order de-duplication.")

epc_cols_to_keep = ['address1', 'postcode', 'uprn', 'total_floor_area', 'current_energy_rating']
if lodge_col:
    epc_cols_to_keep = epc_cols_to_keep + [lodge_col]
epc_chunks = []

print("Extracting Devon records from massive EPC database...")
for chunk in pd.read_csv(epc_csv_path, usecols=epc_cols_to_keep, chunksize=250000, dtype=str, low_memory=False):
    devon_chunk = chunk[chunk['postcode'].isin(devon_postcodes)].copy()

    if not devon_chunk.empty:
        # Normalise EPC address string
        devon_chunk['address1'] = devon_chunk['address1'].str.replace(r'[^\w\s]', '', regex=True).str.replace(' +', ' ', regex=True).str.strip().str.upper()
        epc_chunks.append(devon_chunk)

# Combine chunks, drop duplicates, clear memory
df_epc = pd.concat(epc_chunks, ignore_index=True)
del epc_chunks
gc.collect()

# De-duplicate by address, keeping the MOST RECENT certificate.
# Sort ascending with NaT first so a dated certificate always wins over an undated one,
# then keep='last' retains the latest lodgement date per (address1, postcode).
if lodge_col:
    df_epc[lodge_col] = pd.to_datetime(df_epc[lodge_col], errors='coerce')
    df_epc = df_epc.sort_values(lodge_col, na_position='first')
    df_epc = df_epc.drop_duplicates(subset=['address1', 'postcode'], keep='last').copy()
    df_epc = df_epc.drop(columns=[lodge_col])  # drop date so EPC schema is unchanged downstream
else:
    df_epc = df_epc.drop_duplicates(subset=['address1', 'postcode'], keep='last').copy()
print(f"Isolated {len(df_epc):,} unique EPC records (most recent certificate per address).")

# Execute two-pass bridge merge
print("\nInitiating Pass 1 Merge (Standard Hierarchy)...")
df_res = df_res.merge(
    df_epc,
    left_on=['Match_String_1', 'Postcode'],
    right_on=['address1', 'postcode'],
    how='left'
)

# Split into matched and unmatched
matched_pass1 = df_res[df_res['uprn'].notna()].copy()
unmatched_pass1 = df_res[df_res['uprn'].isna()].copy()

print(f"Pass 1 matched {len(matched_pass1):,} records. Attempting Pass 2 on remaining {len(unmatched_pass1):,} records...")

# Drop the empty EPC columns from the unmatched chunk before second merge
unmatched_pass1 = unmatched_pass1.drop(columns=['address1', 'postcode', 'uprn', 'total_floor_area', 'current_energy_rating'])

# Merge Pass 2 using Swapped String
unmatched_pass1 = unmatched_pass1.merge(
    df_epc,
    left_on=['Match_String_Swapped', 'Postcode'],
    right_on=['address1', 'postcode'],
    how='left'
)

# Recombine the dataset
df_res_final = pd.concat([matched_pass1, unmatched_pass1], ignore_index=True)

# Post-Merge Formatting and Cleanup
df_res_final['Total_Floor_Area'] = pd.to_numeric(df_res_final['total_floor_area'], errors='coerce')
# Natural log of floor area (hedonic size regressor); guard against non-positive values.
# Logged here because floor area is born in this EPC bridge, mirroring the commercial log-at-source pattern.
df_res_final['Log_Floor_Area'] = np.log(df_res_final['Total_Floor_Area'].where(df_res_final['Total_Floor_Area'] > 0))
df_res_final['UPRN'] = df_res_final['uprn']
df_res_final['EPC_Rating'] = df_res_final['current_energy_rating']

matched_count = df_res_final['UPRN'].notna().sum()
match_rate = (matched_count / len(df_res_final)) * 100

print(f"\nBridge Complete: Successfully matched {matched_count:,} total records ({match_rate:.2f}% Match Rate).")

# Retain diagnostic text strings, drop renamed redundant columns
df_res_final = df_res_final.drop(columns=['total_floor_area', 'uprn', 'current_energy_rating', 'postcode', 'address1'])
df_res_final.to_parquet(output_residential_uprn, index=False)
print("Phase 2A dataset safely written to disk with diagnostic strings retained.")

# Update attrition log
print("\nUpdating Central Attrition Log...")
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        attrition_log = json.load(f)
else:
    attrition_log = {}

# Dynamically calculate the match rate breakdown by Property Type
type_breakdown = df_res_final.groupby('Property_Type')['UPRN'].apply(
    lambda x: f"{x.notna().sum()} / {len(x)} ({x.notna().mean()*100:.1f}%)"
).to_dict()

# Append the Phase 2A spatial drop metrics and the diagnostic breakdown
attrition_log["Phase_2A_EPC_Bridge"] = {
    "01_Input_Residential_Rows": len(df_res_final),
    "02_Matched_UPRN_Rows": int(matched_count),
    "03_Match_Rate_Percent": round(match_rate, 2),
    "04_Property_Type_Breakdown": type_breakdown,
}

with open(log_path, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print("Phase 2A Attrition successfully logged with structural caveats!")

Targeting 27,947 unique Devon postcodes for EPC matching.
Extracting Devon records from massive EPC database...
Isolated 346,634 unique EPC records (most recent certificate per address).

Initiating Pass 1 Merge (Standard Hierarchy)...
Pass 1 matched 124,064 records. Attempting Pass 2 on remaining 41,550 records...

Bridge Complete: Successfully matched 124,065 total records (74.91% Match Rate).
Phase 2A dataset safely written to disk with diagnostic strings retained.

Updating Central Attrition Log...
Phase 2A Attrition successfully logged with structural caveats!


In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1A POST-CHECK: TABULAR DATA CHECK
# ==========================================
import pandas as pd
import numpy as np

print("Executing Phase 2A Post-Merge Diagnostic Check...\n")

# Define Path
output_residential_uprn = '/content/drive/MyDrive/MSc_Dissertation_Project/02_processed_data/02_spatial/residential_with_uprn.parquet'

# Load the dataset
df_check = pd.read_parquet(output_residential_uprn)

# Verify Dimensions
print(f"Total Dimensions: {df_check.shape[0]:,} rows and {df_check.shape[1]} columns.")

# Audit Key Spatial Columns (now including the derived Log_Floor_Area)
print("\nMissing Value Audit for New Variables:")
print(df_check[['UPRN', 'Total_Floor_Area', 'EPC_Rating', 'Log_Floor_Area']].isna().sum())

# Log_Floor_Area integrity: present, aligned with floor-area nulls, and reconciling
print("\nLog_Floor_Area Integrity Check:")
if 'Log_Floor_Area' in df_check.columns:
    log_nulls = df_check['Log_Floor_Area'].isna().sum()
    fa_nulls = df_check['Total_Floor_Area'].isna().sum()
    valid = df_check['Total_Floor_Area'] > 0
    reconciles = np.allclose(
        np.exp(df_check.loc[valid, 'Log_Floor_Area']),
        df_check.loc[valid, 'Total_Floor_Area'].astype(float)
    )
    print(f"   -> Column present: True")
    print(f"   -> Null Log_Floor_Area: {log_nulls:,}  (Total_Floor_Area nulls: {fa_nulls:,})")
    print(f"   -> Extra rows blanked by non-positive guard: {int(log_nulls - fa_nulls):,}")
    print(f"   -> exp(Log_Floor_Area) reconciles with Total_Floor_Area: {reconciles}")
else:
    print("   -> WARNING: 'Log_Floor_Area' not found. Re-run Cell 4 (EPC bridge) first.")

# Display the first 5 matched rows to verify structure
print("\nFirst 5 successfully matched rows:")
matched_sample = df_check[df_check['UPRN'].notna()]
print(matched_sample[['Transaction_ID', 'Postcode', 'UPRN', 'Total_Floor_Area', 'Log_Floor_Area', 'EPC_Rating']].head())

# Memory footprint check
memory_usage = df_check.memory_usage(deep=True).sum() / (1024**2)
print(f"\nMemory Usage: {memory_usage:.2f} MB")
print("Tabular data successfully verified. Ready to initiate Phase 2B.")

Executing Phase 2A Post-Merge Diagnostic Check...

Total Dimensions: 165,614 rows and 17 columns.

Missing Value Audit for New Variables:
UPRN                41549
Total_Floor_Area    41393
EPC_Rating          41393
Log_Floor_Area      41393
dtype: int64

Log_Floor_Area Integrity Check:
   -> Column present: True
   -> Null Log_Floor_Area: 41,393  (Total_Floor_Area nulls: 41,393)
   -> Extra rows blanked by non-positive guard: 0
   -> exp(Log_Floor_Area) reconciles with Total_Floor_Area: True

First 5 successfully matched rows:
                           Transaction_ID  Postcode          UPRN  \
0  {4C1B673F-025D-D8EA-E063-4704A8C0BF7D}  EX10 9TP   10093127402   
1  {4C1B673F-025E-D8EA-E063-4704A8C0BF7D}   PL6 5PU  100040500569   
2  {4C1B673F-0261-D8EA-E063-4704A8C0BF7D}   TQ2 7RS  200001111066   
3  {4C1B673F-0262-D8EA-E063-4704A8C0BF7D}   PL7 1TL  100040464929   
4  {4C1B673F-0263-D8EA-E063-4704A8C0BF7D}  EX17 2BP  100040349878   

   Total_Floor_Area  Log_Floor_Area EPC_Rating  
0 

In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1B PRE-CHECK: NSPL POSTCODE LOOKUP
# ==========================================
import pandas as pd
import os

print("Executing Memory-Safe NSPL Database Preview...\n")

BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'

# Updated path matching your exact folder structure
nspl_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/National_Statistics_Postcode_Lookup_NPSL/NSPL_FEB_2026_UK.csv')

try:
    # Input only the first 5 rows to inspect the headers safely without draining Colab RAM
    df_nspl_preview = pd.read_csv(nspl_csv_path, nrows=5)

    print(f"Total Columns Detected: {len(df_nspl_preview.columns)}")
    print("\nExact Column Headers:")
    print(df_nspl_preview.columns.tolist())

    print("\nFirst 5 rows of data:")
    print(df_nspl_preview.head())

except FileNotFoundError:
    print(f"ERROR: File not found at {nspl_csv_path}.")
    print("Please verify the exact file name and ensure it is uploaded to your Google Drive folder.")

Executing Memory-Safe NSPL Database Preview...

Total Columns Detected: 35

Exact Column Headers:
['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'usrtypind', 'east1m', 'north1m', 'gridind', 'oa21cd', 'cty25cd', 'ced25cd', 'lad25cd', 'wd25cd', 'nhser24cd', 'ctry25cd', 'rgn25cd', 'pcon24cd', 'ttwa15cd', 'itl25cd', 'npark16cd', 'lsoa21cd', 'msoa21cd', 'wz11cd', 'sicbl24cd', 'bua24cd', 'ruc21ind', 'oac11ind', 'lat', 'long', 'lep21cd1', 'lep21cd2', 'pfa23cd', 'imd20ind', 'icb23cd']

First 5 rows of data:
      pcd7      pcd8     pcds  dointr  doterm  usrtypind  east1m  north1m  \
0  AB1 0AA  AB1  0AA  AB1 0AA  198001  199606          0  385386   801193   
1  AB1 0AB  AB1  0AB  AB1 0AB  198001  199606          0  385177   801314   
2  AB1 0AD  AB1  0AD  AB1 0AD  198001  199606          0  385053   801092   
3  AB1 0AE  AB1  0AE  AB1 0AE  199402  199606          0  384600   799300   
4  AB1 0AF  AB1  0AF  AB1 0AF  199012  199207          1  384460   800660   

   gridind     oa21cd  ... bua24cd

In [ ]:
# ==========================================
# PART 1 (RESIDENTIAL) - PHASE 1B: NSPL SPATIAL GEOCODING & BUFFER CLIP
# ==========================================
import pandas as pd
import geopandas as gpd
import os
import gc
import json

print("Initiating Part 1, Phase 1B: NSPL Postcode Centroid Geocoding & Buffer Clipping...\n")

# 1. Define Paths
BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'
input_residential = os.path.join(BASE_DIR, '02_processed_data/02_spatial/residential_with_uprn.parquet')
nspl_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/National_Statistics_Postcode_Lookup_NPSL/NSPL_FEB_2026_UK.csv')
buffer_geojson_path = os.path.join(BASE_DIR, '01_raw_data/Stations_and_2000m_Buffer/All_Stations_2000m_Buffer.geojson')
output_parquet = os.path.join(BASE_DIR, '02_processed_data/02_spatial/residential_geocoded.parquet')
log_path = os.path.join(BASE_DIR, '05_logs/script_logs/attrition_log.json')

# 2. Load Residential Data
df_res = pd.read_parquet(input_residential)
initial_row_count = len(df_res)
print(f"Loaded residential dataset: {initial_row_count:,} transactions.")

# Normalise Residential Postcodes
df_res['Postcode_Merge'] = df_res['Postcode'].astype(str).str.replace(' ', '').str.upper()

# 3. Load NSPL Data
print("Loading NSPL database (This may take a moment)...")
df_nspl = pd.read_csv(nspl_csv_path, usecols=['pcds', 'east1m', 'north1m'])
df_nspl['pcds_Merge'] = df_nspl['pcds'].astype(str).str.replace(' ', '').str.upper()
df_nspl = df_nspl.drop_duplicates(subset=['pcds_Merge'])

# 4. Merge
print("Executing spatial bridge merge...")
df_merged = df_res.merge(
    df_nspl[['pcds_Merge', 'east1m', 'north1m']],
    left_on='Postcode_Merge',
    right_on='pcds_Merge',
    how='left'
)

# Convert to spatial geometry
print("Converting exact coordinates to spatial point geometries...")
gdf_res = gpd.GeoDataFrame(
    df_merged,
    geometry=gpd.points_from_xy(df_merged['east1m'], df_merged['north1m']),
    crs="EPSG:27700"
)

# Clean up merging artifacts
gdf_res = gdf_res.drop(columns=['Postcode_Merge', 'pcds_Merge'])
matched_points = gdf_res['east1m'].notna().sum()
print(f"Geocoding successful for {matched_points:,} properties.")

# 5. Apply the 2000m Station Buffer Spatial Filter
print(f"\nLoading 2000m station buffer from: {os.path.basename(buffer_geojson_path)}...")
gdf_buffer = gpd.read_file(buffer_geojson_path)

if gdf_buffer.crs != "EPSG:27700":
    gdf_buffer = gdf_buffer.to_crs("EPSG:27700")

print("Clipping residential points to the 2000m buffer zone...")
gdf_res_valid = gdf_res[gdf_res['geometry'].notna()]
gdf_final = gpd.sjoin(gdf_res_valid, gdf_buffer[['geometry']], how='inner', predicate='within')

if 'index_right' in gdf_final.columns:
    gdf_final = gdf_final.drop(columns=['index_right'])

final_count = len(gdf_final)
retention_rate = (final_count / initial_row_count) * 100

print(f"\nSpatial Filtering Complete: {final_count:,} residential properties successfully isolated.")
print(f"Note: The retention rate of {retention_rate:.2f}% reflects properties within the defined 2000m catchment areas.")

# 6. Save result
gdf_final.to_parquet(output_parquet, index=False)
print("Part 1 complete. GeoParquet safely written to disk (residential_geocoded.parquet).")

# 7. Update Central Attrition Log
print("\nUpdating Central Attrition Log...")
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        attrition_log = json.load(f)
else:
    attrition_log = {}

attrition_log["Part_1_Phase_1B_NSPL_Geocoding_and_Clipping"] = {
    "01_Input_Residential_Rows": initial_row_count,
    "02_Geocoded_Point_Rows": int(matched_points),
    "03_Final_Buffered_Rows": int(final_count),
    "04_Retention_Rate_Percent": round(retention_rate, 2),
    "Note": "Residential dataset geocoded using NSPL centroids and spatially clipped to 2000m rail station buffers. Attrition reflects the exclusion of properties outside station catchments to ensure spatial focus. CRS locked to EPSG:27700."
}

with open(log_path, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print("Part 1 - Phase 1B Attrition successfully logged!")

# Flush memory
del df_res, df_nspl, df_merged, gdf_res, gdf_buffer, gdf_res_valid, gdf_final
gc.collect()

Initiating Part 1, Phase 1B: NSPL Postcode Centroid Geocoding & Buffer Clipping...

Loaded residential dataset: 165,614 transactions.
Loading NSPL database (This may take a moment)...
Executing spatial bridge merge...
Converting exact coordinates to spatial point geometries...
Geocoding successful for 165,576 properties.

Loading 2000m station buffer from: All_Stations_2000m_Buffer.geojson...
Clipping residential points to the 2000m buffer zone...

Spatial Filtering Complete: 43,413 residential properties successfully isolated.
Note: The retention rate of 26.21% reflects properties within the defined 2000m catchment areas.
Part 1 complete. GeoParquet safely written to disk (residential_geocoded.parquet).

Updating Central Attrition Log...
Part 1 - Phase 1B Attrition successfully logged!


33

In [ ]:
# ==========================================
# PART 2 (RESIDENTIAL) - PHASE 2A POST-CHECK: GEOMETRY DIAGNOSTIC
# ==========================================
import geopandas as gpd
import os

print("Executing Residential Phase 2A Post-Merge Diagnostic Check...\n")

# Define Path
BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'
# Ensure this matches the output filename from your NSPL geocoding script
output_residential_geoparquet = os.path.join(BASE_DIR, '02_processed_data/02_spatial/residential_geocoded.parquet')

# Load the dataset
gdf_res_check = gpd.read_parquet(output_residential_geoparquet)

# Verify Dimensions
print(f"Total Residential Dimensions: {gdf_res_check.shape[0]:,} rows and {gdf_res_check.shape[1]} columns.")

# Audit Key Spatial Anchors
# We check the columns that came from your NSPL merge
print("\nMissing Value Audit for Spatial Anchors:")
spatial_cols = [c for c in ['east1m', 'north1m', 'geometry'] if c in gdf_res_check.columns]
print(gdf_res_check[spatial_cols].isna().sum())

# Display the first 5 rows to verify structure
print("\nFirst 5 successfully matched rows (Geometry present):")
display_cols = [col for col in ['Postcode', 'east1m', 'north1m', 'geometry'] if col in gdf_res_check.columns]
print(gdf_res_check[display_cols].head())

# Verify Coordinate Reference System (CRS)
print(f"\nActive CRS: {gdf_res_check.crs}")

# Memory footprint check
memory_usage = gdf_res_check.memory_usage(deep=True).sum() / (1024**2)
print(f"\nMemory Usage: {memory_usage:.2f} MB")

# Final verification of the clip result
station_col = [c for c in gdf_res_check.columns if 'station' in c.lower() or 'index_right' in c]
if station_col:
    print(f"\nSpatial Join verified against: {station_col[0]}")
else:
    print("\nWarning: Could not detect station join column. Check your spatial join inputs.")

print("\nResidential GeoParquet point geometry verified. The residential spatial pipeline is complete!")

Executing Residential Phase 2A Post-Merge Diagnostic Check...

Total Residential Dimensions: 43,413 rows and 20 columns.

Missing Value Audit for Spatial Anchors:
east1m      0
north1m     0
geometry    0
dtype: int64

First 5 successfully matched rows (Geometry present):
   Postcode    east1m   north1m               geometry
0   TQ2 7RS  288799.0   65748.0   POINT (288799 65748)
1  EX17 2BP  283312.0  100411.0  POINT (283312 100411)
2   EX4 1SR  290109.0   92716.0   POINT (290109 92716)
3   PL1 5BX  247491.0   55173.0   POINT (247491 55173)
4  EX31 3RH  254190.0  132019.0  POINT (254190 132019)

Active CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "OSGB36 / British National Grid", "base_crs": {"name": "OSGB36", "datum": {"type": "GeodeticReferenceFrame", "name": "Ordnance Survey of Great Britain 1936", "ellipsoid": {"name": "Airy 1830", "semi_major_axis": 6377563.396, "inverse_flattening": 299.3249646}}, "coordinate_system": {"s

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) - PHASE 2A PRE-CHECK: NSPL POSTCODE LOOKUP
# ==========================================
import pandas as pd
import os

print("Executing Memory-Safe NSPL Database Preview...\n")

BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'

# Updated path matching your exact folder structure
nspl_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/National_Statistics_Postcode_Lookup_NPSL/NSPL_FEB_2026_UK.csv')

try:
    # Input only the first 5 rows to inspect the headers safely without draining Colab RAM
    df_nspl_preview = pd.read_csv(nspl_csv_path, nrows=5)

    print(f"Total Columns Detected: {len(df_nspl_preview.columns)}")
    print("\nExact Column Headers:")
    print(df_nspl_preview.columns.tolist())

    print("\nFirst 5 rows of data:")
    print(df_nspl_preview.head())

except FileNotFoundError:
    print(f"ERROR: File not found at {nspl_csv_path}.")
    print("Please verify the exact file name and ensure it is uploaded to your Google Drive folder.")

Executing Memory-Safe NSPL Database Preview...

Total Columns Detected: 35

Exact Column Headers:
['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'usrtypind', 'east1m', 'north1m', 'gridind', 'oa21cd', 'cty25cd', 'ced25cd', 'lad25cd', 'wd25cd', 'nhser24cd', 'ctry25cd', 'rgn25cd', 'pcon24cd', 'ttwa15cd', 'itl25cd', 'npark16cd', 'lsoa21cd', 'msoa21cd', 'wz11cd', 'sicbl24cd', 'bua24cd', 'ruc21ind', 'oac11ind', 'lat', 'long', 'lep21cd1', 'lep21cd2', 'pfa23cd', 'imd20ind', 'icb23cd']

First 5 rows of data:
      pcd7      pcd8     pcds  dointr  doterm  usrtypind  east1m  north1m  \
0  AB1 0AA  AB1  0AA  AB1 0AA  198001  199606          0  385386   801193   
1  AB1 0AB  AB1  0AB  AB1 0AB  198001  199606          0  385177   801314   
2  AB1 0AD  AB1  0AD  AB1 0AD  198001  199606          0  385053   801092   
3  AB1 0AE  AB1  0AE  AB1 0AE  199402  199606          0  384600   799300   
4  AB1 0AF  AB1  0AF  AB1 0AF  199012  199207          1  384460   800660   

   gridind     oa21cd  ... bua24cd

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) - PHASE 2A: NSPL GEOCODING & BUFFER FILTER
# ==========================================
import pandas as pd
import geopandas as gpd
import os
import gc
import json

print("Initiating Part 2, Phase 2A: NSPL Postcode Centroid Geocoding & Buffer Clipping...\n")

# Define Paths
BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'
input_commercial = os.path.join(BASE_DIR, '02_processed_data/01_cleaned/commercial_cleaned.parquet')
nspl_csv_path = os.path.join(BASE_DIR, '01_raw_data/spatial/National_Statistics_Postcode_Lookup_NPSL/NSPL_FEB_2026_UK.csv')
buffer_geojson_path = os.path.join(BASE_DIR, '01_raw_data/Stations_and_2000m_Buffer/All_Stations_2000m_Buffer.geojson')
output_geoparquet = os.path.join(BASE_DIR, '02_processed_data/02_spatial/commercial_geocoded.parquet')
log_path = os.path.join(BASE_DIR, '05_logs/script_logs/attrition_log.json')

# Load Commercial Data
df_com = pd.read_parquet(input_commercial)
initial_row_count = len(df_com) # Capture the count before any memory flushes!
print(f"Loaded commercial dataset: {initial_row_count:,} transactions.")

# Normalise Commercial Postcodes (Strip all spaces and uppercase)
df_com['Postcode_Merge'] = df_com['Postcode'].astype(str).str.replace(' ', '').str.upper()

# Load NSPL Data (Memory Safe - only input required columns)
print("Loading NSPL database (This may take a moment)...")
df_nspl = pd.read_csv(nspl_csv_path, usecols=['pcds', 'east1m', 'north1m'])

# Normalise NSPL Postcodes to match
df_nspl['pcds_Merge'] = df_nspl['pcds'].astype(str).str.replace(' ', '').str.upper()
df_nspl = df_nspl.drop_duplicates(subset=['pcds_Merge'])

# Execute the Merge
print("Executing spatial bridge merge...")
df_merged = df_com.merge(
    df_nspl[['pcds_Merge', 'east1m', 'north1m']],
    left_on='Postcode_Merge',
    right_on='pcds_Merge',
    how='left'
)

# Convert to spatial geometry (Point Centroids)
print("Converting exact coordinates to spatial point geometries...")
gdf_com = gpd.GeoDataFrame(
    df_merged,
    geometry=gpd.points_from_xy(df_merged['east1m'], df_merged['north1m']),
    crs="EPSG:27700"
)

# Clean up merging artifacts
gdf_com = gdf_com.drop(columns=['Postcode_Merge', 'pcds_Merge'])
matched_points = gdf_com['east1m'].notna().sum()
print(f"Geocoding successful for {matched_points:,} properties.")

# Apply the 2000m Station Buffer Spatial Filter
print(f"\nLoading 2000m station buffer from: {os.path.basename(buffer_geojson_path)}...")
gdf_buffer = gpd.read_file(buffer_geojson_path)

# Ensure the buffer is strictly using EPSG:27700 to match our points
if gdf_buffer.crs != "EPSG:27700":
    gdf_buffer = gdf_buffer.to_crs("EPSG:27700")

print("Clipping commercial points to the 2000m buffer zone...")
# Drop rows without valid geometries before attempting the spatial join
gdf_com_valid = gdf_com[gdf_com['geometry'].notna()]

# Execute an inner spatial join to keep strictly the points inside the buffer polygons
gdf_final = gpd.sjoin(gdf_com_valid, gdf_buffer[['geometry']], how='inner', predicate='within')

# Clean up the artifact index column created by sjoin
if 'index_right' in gdf_final.columns:
    gdf_final = gdf_final.drop(columns=['index_right'])

final_count = len(gdf_final)
retention_rate = (final_count / initial_row_count) * 100

print(f"\nSpatial Filtering Complete: {final_count:,} commercial properties successfully isolated.")
print(f"Note: The intentional drop to a {retention_rate:.2f}% retention rate is by design. Properties outside the 2000m station buffers were systematically dropped to isolate the treatment/control zones and eliminate rural market noise, maintaining methodological consistency with the residential pipeline.")

# Save as a GeoParquet
gdf_final.to_parquet(output_geoparquet, index=False)
print("Part 2 complete. GeoParquet safely written to disk (commercial_geocoded.parquet).")

# Update Central Attrition Log
print("\nUpdating Central Attrition Log...")
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        attrition_log = json.load(f)
else:
    attrition_log = {}

attrition_log["Part_2_Phase_2A_NSPL_Geocoding_and_Clipping"] = {
    "01_Input_Commercial_Rows": initial_row_count,
    "02_Geocoded_Point_Rows": int(matched_points),
    "03_Final_Buffered_Rows": int(final_count),
    "04_Retention_Rate_Percent": round(retention_rate, 2),
    "Note": "Commercial dataset geocoded using NSPL point centroids, then explicitly spatially clipped via inner join. The attrition is an intentional methodological filter to strictly retain properties within 2000m station buffers, mirroring the residential spatial constraints and eliminating extraneous rural market observations. CRS locked to EPSG:27700."
}

with open(log_path, 'w') as f:
    json.dump(attrition_log, f, indent=4)

print("Part 2 - Phase 2A Attrition successfully logged!")

# Flush memory safely at the very end
del df_com, df_nspl, df_merged, gdf_com, gdf_buffer, gdf_com_valid
gc.collect()

Initiating Part 2, Phase 2A: NSPL Postcode Centroid Geocoding & Buffer Clipping...

Loaded commercial dataset: 124,797 transactions.
Loading NSPL database (This may take a moment)...
Executing spatial bridge merge...
Converting exact coordinates to spatial point geometries...
Geocoding successful for 124,298 properties.

Loading 2000m station buffer from: All_Stations_2000m_Buffer.geojson...
Clipping commercial points to the 2000m buffer zone...

Spatial Filtering Complete: 28,361 commercial properties successfully isolated.
Note: The intentional drop to a 22.73% retention rate is by design. Properties outside the 2000m station buffers were systematically dropped to isolate the treatment/control zones and eliminate rural market noise, maintaining methodological consistency with the residential pipeline.
Part 2 complete. GeoParquet safely written to disk (commercial_geocoded.parquet).

Updating Central Attrition Log...
Part 2 - Phase 2A Attrition successfully logged!


77

In [ ]:
# ==========================================
# PART 2 (COMMERCIAL) - PHASE 2A POST-CHECK: GEOMETRY DIAGNOSTIC
# ==========================================
import geopandas as gpd
import os

print("Executing Phase 2A Post-Merge Diagnostic Check...\n")

# Define Path
BASE_DIR = '/content/drive/MyDrive/MSc_Dissertation_Project'
output_geoparquet = os.path.join(BASE_DIR, '02_processed_data/02_spatial/commercial_geocoded.parquet')

# Load the dataset
gdf_check = gpd.read_parquet(output_geoparquet)

# Verify Dimensions
print(f"Total Dimensions: {gdf_check.shape[0]:,} rows and {gdf_check.shape[1]} columns.")

# Audit Key Spatial Anchors
print("\nMissing Value Audit for Spatial Anchors:")
print(gdf_check[['east1m', 'north1m', 'geometry']].isna().sum())

# Display the first 5 rows to verify structure
print("\nFirst 5 successfully matched rows (Geometry present):")

# Dynamically select columns just in case your commercial data has different headers
display_cols = [col for col in ['Postcode', 'east1m', 'north1m', 'geometry'] if col in gdf_check.columns]
print(gdf_check[display_cols].head())

# Verify Coordinate Reference System (CRS)
print(f"\nActive CRS: {gdf_check.crs}")

# Memory footprint check
memory_usage = gdf_check.memory_usage(deep=True).sum() / (1024**2)
print(f"\nMemory Usage: {memory_usage:.2f} MB")
print("\nCommercial GeoParquet point geometry verified. The entire Spatial Bridge phase is complete!")

Executing Phase 2A Post-Merge Diagnostic Check...

Total Dimensions: 28,361 rows and 11 columns.

Missing Value Audit for Spatial Anchors:
east1m      0
north1m     0
geometry    0
dtype: int64

First 5 successfully matched rows (Geometry present):
   Postcode    east1m  north1m              geometry
0  PL12 4EA  243013.0  58756.0  POINT (243013 58756)
1  PL12 4EB  243154.0  58760.0  POINT (243154 58760)
2  PL12 4EB  243154.0  58760.0  POINT (243154 58760)
3  PL12 6AN  243078.0  58840.0  POINT (243078 58840)
4  PL11 2JW  243775.0  55294.0  POINT (243775 55294)

Active CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "OSGB36 / British National Grid", "base_crs": {"name": "OSGB36", "datum": {"type": "GeodeticReferenceFrame", "name": "Ordnance Survey of Great Britain 1936", "ellipsoid": {"name": "Airy 1830", "semi_major_axis": 6377563.396, "inverse_flattening": 299.3249646}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"n